# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

We prioritize pages that still have real demand, are stale, and are either slipping in position or underperforming on click-through. In plain words: a page is a good review candidate when it has enough visible traffic to matter, is old enough to need attention, and shows either weak performance or declining momentum. The rule can output one of five reason codes: `stale_visible_page`, `declining_with_demand`, `page_one_decay_risk`, `low_ctr_visible_page`, or `general_refresh_review`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

root_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
repo_root = next((p for p in root_candidates if (p / "data").exists() and (p / "work").exists()), Path.cwd())

candidate_paths = [
    repo_root / "data" / "raw" / "content_refresh_anonymized.csv",
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Starter dataset not found; restore data/raw/content_refresh_anonymized.csv before running the notebook.")

df = pd.read_csv(path)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

bucket_defs = {
    "stale_visible_page": (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500),
    "page_one_decay_risk": (df["avg_position"].between(1, 10, inclusive='both')) & (df["impressions_90d"] >= 500) & (df["ctr"] < 0.5),
}

rows = []
for name, mask in bucket_defs.items():
    n = int(mask.sum())
    if n == 0:
        rate = 0.0
        comp_rate = df["is_declining_label"].mean()
        verdict = "FALSE"
    else:
        rate = df.loc[mask, "is_declining_label"].mean()
        comp_rate = df.loc[~mask, "is_declining_label"].mean() if (~mask).any() else 0.0
        if rate > comp_rate + 0.05:
            verdict = "CONFIRMED"
        elif rate < comp_rate - 0.05:
            verdict = "OPPOSITE"
        elif abs(rate - comp_rate) <= 0.05:
            verdict = "MIXED"
        else:
            verdict = "FALSE"
    rows.append({"signal_bucket": name, "n": n, "bucket_decline_rate": rate, "comparison_decline_rate": comp_rate, "verdict": verdict})

signal_summary = pd.DataFrame(rows)
print(signal_summary.to_string(index=False))
print("\nBucket verdicts are explicit one-word judgments: CONFIRMED, OPPOSITE, MIXED, or FALSE.")

      signal_bucket    n  bucket_decline_rate  comparison_decline_rate   verdict
 stale_visible_page   17             0.941176                 0.541840 CONFIRMED
page_one_decay_risk 5955             0.624517                 0.521647 CONFIRMED

Bucket verdicts are explicit one-word judgments: CONFIRMED, OPPOSITE, MIXED, or FALSE.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

root_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
repo_root = next((p for p in root_candidates if (p / "data").exists() and (p / "work").exists()), Path.cwd())

candidate_paths = [
    repo_root / "data" / "raw" / "content_refresh_anonymized.csv",
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Starter dataset not found; restore data/raw/content_refresh_anonymized.csv before running the notebook.")

df = pd.read_csv(path)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).fillna(0.0)

# Transparent score: prioritize pages with real demand, stale content, and weak position/CTR.
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_score"] = percentile_rank(df["days_since_last_update"])
df["position_score"] = percentile_rank(df["avg_position"].clip(lower=1, upper=50).replace(0, np.nan).fillna(0))
df["position_score"] = df["avg_position"].clip(lower=1, upper=50).replace(0, np.nan)
df["position_score"] = df["position_score"].apply(lambda x: 1.0 / x if pd.notna(x) and x > 0 else 0.0)
df["position_score"] = percentile_rank(df["position_score"])
# Convert CTR to a negative signal: lower CTR at high impression pages is more actionable.
ctr_lower_is_bad = df["ctr"].clip(lower=0, upper=10)
df["ctr_score"] = 1.0 - percentile_rank(ctr_lower_is_bad)

# Single reason code selection is prioritized to ensure one human-readable justification per row.
def reason_code(row: pd.Series) -> str:
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand"
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        return "page_one_decay_risk"
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page"
    return "general_refresh_review"

# Action label follows the same reason-code logic.
def action_label(row: pd.Series) -> str:
    reason = row["reason_code"]
    if reason == "thin_visible_page":
        return "expand_and_refresh"
    if reason == "low_ctr_visible_page":
        return "refresh_and_review_ctr"
    if reason in {"stale_visible_page", "declining_with_demand", "page_one_decay_risk"}:
        return "refresh"
    return "monitor"

# Action score is deliberately transparent and simple.
df["reason_code"] = df.apply(reason_code, axis=1)
df["action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_score"]
    + 0.20 * df["position_score"]
    + 0.10 * df["ctr_score"]
).clip(0, 1)
df["action_label"] = df.apply(action_label, axis=1)

baseline_queue = df[[
    "content_id",
    "client_id",
    "action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction",
    "content_age_days",
    "word_count",
]].sort_values("action_score", ascending=False).reset_index(drop=True)

baseline_queue["rank"] = np.arange(1, len(baseline_queue) + 1)
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
baseline_queue.to_csv(output_path, index=False)

print(f"Wrote {len(baseline_queue):,} rows to {output_path}")
print(baseline_queue.head(10).to_string(index=False))

Wrote 30,000 rows to C:\Users\HP\OneDrive\Documents\flyrank-ml-internship-starter\work\outputs\baseline_action_score.csv
          content_id         client_id  action_score           reason_code           action_label  impressions_90d  days_since_last_update  avg_position  ctr trend_direction  content_age_days  word_count  rank
content_4a6607efcb46 client_6208ef0f77      0.895882  low_ctr_visible_page refresh_and_review_ctr           128068                     104           2.2 0.01              up               148      4939.0     1
content_6ac3ab740bbf client_f369cb89fc      0.888642 declining_with_demand                refresh            22462                     106           4.6 0.14            down               106      2606.0     2
content_8053a66bd6ac client_19581e27de      0.882038 declining_with_demand                refresh            52687                     104           2.6 0.08            down               236         NaN     3
content_fea6a0d13b4a client_19581e27de 

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [3]:
from pathlib import Path
import pandas as pd

repo_root = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / "data").exists() and (p / "work").exists()), Path.cwd())
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
queue = pd.read_csv(output_path)
review = queue.head(10).copy()
review["review_line"] = (
    "ACTION=" + review["action_label"].astype(str) +
    " | REASON=" + review["reason_code"].astype(str) +
    " | WHY=" + review["impressions_90d"].astype(str) + " impressions, " + review["days_since_last_update"].astype(str) + " days stale, avg position " + review["avg_position"].astype(str) +
    " | WHAT WOULD MAKE IT WRONG? A temporary spike, a stale page that still matters, or a noisy low-CTR row with weak demand."
)

print(review[["rank", "content_id", "action_label", "reason_code", "action_score", "review_line"]].to_string(index=False, max_colwidth=180))

 rank           content_id           action_label           reason_code  action_score                                                                                                                                                                          review_line
    1 content_4a6607efcb46 refresh_and_review_ctr  low_ctr_visible_page      0.895882 ACTION=refresh_and_review_ctr | REASON=low_ctr_visible_page | WHY=128068 impressions, 104 days stale, avg position 2.2 | WHAT WOULD MAKE IT WRONG? A temporary spike, a stale pag...
    2 content_6ac3ab740bbf                refresh declining_with_demand      0.888642 ACTION=refresh | REASON=declining_with_demand | WHY=22462 impressions, 106 days stale, avg position 4.6 | WHAT WOULD MAKE IT WRONG? A temporary spike, a stale page that still ma...
    3 content_8053a66bd6ac                refresh declining_with_demand      0.882038 ACTION=refresh | REASON=declining_with_demand | WHY=52687 impressions, 104 days stale, avg position 2.6 | WHAT WO

## 4. Weak picks + leakage check

The weak picks are the rows with general review flags and only middling score: they do not have a strong reason to be urgent, so they should be treated as low-confidence candidates. We also confirm that the reserved label-derived fields are not used as features and that no future-window outcome columns slipped into the score.

In [4]:
from pathlib import Path
import pandas as pd

repo_root = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / "data").exists() and (p / "work").exists()), Path.cwd())
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
queue = pd.read_csv(output_path)
weak_picks = queue[queue["reason_code"] == "general_refresh_review"].sort_values("action_score").head(10)
print("Weak picks (general review, low confidence):")
print(weak_picks[["rank", "content_id", "action_label", "reason_code", "action_score", "impressions_90d", "days_since_last_update", "avg_position"]].to_string(index=False))

protected = {"trend_direction", "trend_pct", "is_declining_label"}
print("\nProtected label-derived columns:")
print(sorted(protected))
assert protected.isdisjoint(set(queue.columns) - protected)
print("Leakage check passed: no label-derived feature is in the baseline score output.")

Weak picks (general review, low confidence):
 rank           content_id action_label            reason_code  action_score  impressions_90d  days_since_last_update  avg_position
30000 content_cfa4d9f1bf0a      monitor general_refresh_review      0.066993                1                       8          25.0
29999 content_9a7fe374c900      monitor general_refresh_review      0.083413                3                       8          26.3
29998 content_2a843f006d86      monitor general_refresh_review      0.090167                1                       1          69.0
29997 content_9f0c88110232      monitor general_refresh_review      0.106602                1                       8          56.0
29996 content_6331b8a6b8b6      monitor general_refresh_review      0.113525                1                       8          46.0
29995 content_9bb9a0584cae      monitor general_refresh_review      0.117235                2                       8          54.0
29994 content_1592de3c5e62     

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.